# 01 · Data Loading

Load the four raw input files (`data_matrix.csv`, `feature_metadata.csv`, 
`acquisition_list.csv`, `exogenous_standards.csv`), inspect their shapes and contents, 
verify that samples and features are aligned across files, and save a consolidated object for downstream notebooks.

**Input:** `files/data/input/`  
**Output:** `files/data/processed/raw_aligned.pkl`

In [1]:
import importlib
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

np.random.seed(42)

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR = Path("../files/data/input")
OUT_DIR  = Path("../files/data/processed")
OUT_DIR.mkdir(exist_ok=True)

# ── Package versions ───────────────────────────────────────────────────────
for pkg in ["numpy", "pandas", "scipy", "sklearn", "matplotlib", "seaborn"]:
    try:
        mod = importlib.import_module(pkg)
        print(f"{pkg:15s} {mod.__version__}")
    except ModuleNotFoundError:
        print(f"{pkg:15s} not installed")

numpy           2.2.1
pandas          2.2.3
scipy           1.14.1


sklearn         1.6.0


matplotlib      3.10.0


seaborn         0.13.2


## 1 · Load Raw Input Files

We read the four input CSVs directly from `files/data/input/`. No transformation is applied at this stage — this is raw data as produced by the instrument pipeline.

| File | Description |
|------|-------------|
| `data_matrix.csv` | N × M matrix of LC-MS peak intensities (rows = samples, cols = features) |
| `feature_metadata.csv` | m/z and retention time statistics for each feature |
| `acquisition_list.csv` | Sample metadata: class label, batch, run order |
| `exogenous_standards.csv` | Known spiked-in glycan standards used to verify instrument calibration |

In [2]:
data_matrix  = pd.read_csv(DATA_DIR / "data_matrix.csv",       index_col="sample")
feature_meta = pd.read_csv(DATA_DIR / "feature_metadata.csv",  index_col="feature")
sample_meta  = pd.read_csv(DATA_DIR / "acquisition_list.csv",  index_col="sample")
standards    = pd.read_csv(DATA_DIR / "exogenous_standards.csv")

print(f"data_matrix  : {data_matrix.shape[0]:>4d} samples  × {data_matrix.shape[1]:>4d} features")
print(f"feature_meta : {feature_meta.shape[0]:>4d} features × {feature_meta.shape[1]:>4d} columns")
print(f"sample_meta  : {sample_meta.shape[0]:>4d} samples  × {sample_meta.shape[1]:>4d} columns")
print(f"standards    : {standards.shape[0]:>4d} rows     × {standards.shape[1]:>4d} columns")

data_matrix  :  124 samples  ×  252 features
feature_meta :  252 features ×   10 columns
sample_meta  :  124 samples  ×    4 columns
standards    :    4 rows     ×    3 columns


## 2 · Inspect Contents

Preview each DataFrame to confirm column names, data types, and value ranges look sensible.

In [3]:
print("─── data_matrix (first 4 samples, first 6 features) ───")
display(data_matrix.iloc[:4, :6])

print("\n─── feature_meta ───")
display(feature_meta.head(4))

print("\n─── sample_meta ───")
display(sample_meta.head(8))

print("\n─── exogenous_standards ───")
display(standards)

─── data_matrix (first 4 samples, first 6 features) ───


,FT-000,FT-001,FT-002,FT-003,FT-004,FT-005
sample,,,,,,
20241106-297-Blank1,0.0,0.000000,0.0,0.000000,0.0,0.0
20241106-297-Blank2,0.0,0.000000,0.0,5.770641,0.0,0.0
20241106-297-Blank3,0.0,0.000000,0.0,0.000000,0.0,0.0
20241106-297-Blank4,0.0,5.497865,0.0,0.000000,0.0,0.0



─── feature_meta ───


,mz,mz max,mz min,mz std,rt,rt end,rt max,rt min,rt start,rt std
feature,,,,,,,,,,
FT-000,359.108768,359.130591,359.090122,0.010955,543.846374,561.270879,546.697803,539.941156,528.714227,1.683901
FT-001,357.109546,357.130018,357.091871,0.010668,543.839157,563.021530,546.948017,540.038924,528.530311,1.662290
FT-002,889.649931,889.702216,889.601547,0.025552,609.099742,627.717251,611.746501,606.426553,596.789046,1.324647
FT-003,355.111928,355.132772,355.093653,0.010508,543.995269,566.951285,547.164503,540.681882,526.943213,1.609071



─── sample_meta ───


,class,id,order,batch
sample,,,,
20241106-297-Blank1,B,20241106-297-Blank1,1,1
20241106-297-Blank2,B,20241106-297-Blank2,2,1
20241106-297-Blank3,B,20241106-297-Blank3,3,1
20241106-297-Blank4,B,20241106-297-Blank4,102,1
20241106-297-Blank5,B,20241106-297-Blank5,103,1
20241106-297-Blank6,B,20241106-297-Blank6,104,1
20241106-297-Diluted-QC1,dQC,20241106-297-Diluted-QC1,7,1
20241106-297-Diluted-QC2,dQC,20241106-297-Diluted-QC2,8,1



─── exogenous_standards ───


,compound_id,mz,Retention_time
0,GU4,886.40,610
1,GU5,1048.45,753
2,GU14,1253.96,1500
3,GU15,1334.99,1551


In [4]:
# Value range in the data matrix (non-zero only)
nonzero = data_matrix.values[data_matrix.values != 0]
print(f"data_matrix value range (non-zero entries):")
print(f"  min    : {nonzero.min():.4f}")
print(f"  max    : {nonzero.max():.4f}")
print(f"  median : {np.median(nonzero):.4f}")
print()
print(f"feature_meta columns : {list(feature_meta.columns)}")
print(f"sample_meta columns  : {list(sample_meta.columns)}")
print(f"m/z range (feature_meta): {feature_meta['mz'].min():.2f} – {feature_meta['mz'].max():.2f} Da")
print(f"RT range  (feature_meta): {feature_meta['rt'].min():.1f} – {feature_meta['rt'].max():.1f} s")

data_matrix value range (non-zero entries):
  min    : 0.1126
  max    : 55518.0492
  median : 1304.6215

feature_meta columns : ['mz', 'mz max', 'mz min', 'mz std', 'rt', 'rt end', 'rt max', 'rt min', 'rt start', 'rt std']
sample_meta columns  : ['class', 'id', 'order', 'batch']
m/z range (feature_meta): 293.18 – 1846.27 Da
RT range  (feature_meta): 543.8 – 1551.4 s


## 3 · Verify Alignment

Before any analysis we confirm that:
1. **Sample alignment** — every row in `data_matrix` has a matching entry in `sample_meta`, and vice versa.
2. **Feature alignment** — every column in `data_matrix` has a matching entry in `feature_meta`, and vice versa.

Mismatches here would indicate a data wrangling error upstream and would silently corrupt all downstream analyses.

In [5]:
dm_samples   = set(data_matrix.index)
meta_samples = set(sample_meta.index)

only_in_dm   = sorted(dm_samples - meta_samples)
only_in_meta = sorted(meta_samples - dm_samples)

print("── Sample alignment ─────────────────────────────────────────")
print(f"  Rows in data_matrix : {len(dm_samples)}")
print(f"  Rows in sample_meta : {len(meta_samples)}")
print(f"  Shared              : {len(dm_samples & meta_samples)}")
if only_in_dm:
    print(f"  ⚠ Only in data_matrix : {only_in_dm}")
if only_in_meta:
    print(f"  ⚠ Only in sample_meta : {only_in_meta}")
if not only_in_dm and not only_in_meta:
    print("  ✓ Perfect match — all sample IDs align")

dm_features   = set(data_matrix.columns)
meta_features = set(feature_meta.index)

only_in_dm_f   = sorted(dm_features - meta_features)
only_in_meta_f = sorted(meta_features - dm_features)

print("\n── Feature alignment ────────────────────────────────────────")
print(f"  Columns in data_matrix  : {len(dm_features)}")
print(f"  Rows in feature_meta    : {len(meta_features)}")
print(f"  Shared                  : {len(dm_features & meta_features)}")
if only_in_dm_f:
    print(f"  ⚠ Only in data_matrix   : {only_in_dm_f[:10]}{'…' if len(only_in_dm_f)>10 else ''}")
if only_in_meta_f:
    print(f"  ⚠ Only in feature_meta  : {only_in_meta_f[:10]}{'…' if len(only_in_meta_f)>10 else ''}")
if not only_in_dm_f and not only_in_meta_f:
    print("  ✓ Perfect match — all feature IDs align")

# Reindex both matrices to the common sets (defensive alignment)
common_samples  = sorted(dm_samples & meta_samples)
common_features = sorted(dm_features & meta_features, key=lambda x: int(x.split("-")[1]))

data_matrix  = data_matrix.loc[common_samples, common_features]
feature_meta = feature_meta.loc[common_features]
sample_meta  = sample_meta.loc[common_samples]

print(f"\nAligned data_matrix : {data_matrix.shape}")

── Sample alignment ─────────────────────────────────────────
  Rows in data_matrix : 124
  Rows in sample_meta : 124
  Shared              : 124
  ✓ Perfect match — all sample IDs align

── Feature alignment ────────────────────────────────────────
  Columns in data_matrix  : 252
  Rows in feature_meta    : 252
  Shared                  : 252
  ✓ Perfect match — all feature IDs align

Aligned data_matrix : (124, 252)


## 4 · Sample Summary

Summarise the cohort design: how many samples per class and per batch, and how the acquisition was spread across run order.

**Class key:**

| Code | Biological group |
|------|-----------------|
| `French` | Lung cancer patients (Disease cohort) |
| `LMU` | Benign pulmonary disease patients |
| `Dunn` | Healthy volunteers (Control cohort) |
| `QC` | Pooled quality control (repeated injections) |
| `dQC` | Diluted QC (linearity / dynamic range check) |
| `B` | Solvent blank (background / contamination check) |
| `SS` | System suitability (instrument performance check) |

In [6]:
CLASS_LABEL = {
    "French": "Disease (lung cancer)",
    "LMU":    "Benign (pulmonary disease)",
    "Dunn":   "Control (healthy)",
    "QC":     "Pooled QC",
    "dQC":    "Diluted QC",
    "B":      "Blank",
    "SS":     "System suitability",
}

print("── Samples per class ────────────────────────────────────────")
class_counts = sample_meta["class"].value_counts()
for cls, n in class_counts.items():
    label = CLASS_LABEL.get(cls, cls)
    print(f"  {cls:8s}  {label:<30s}  n = {n}")

print(f"\n  Total: {len(sample_meta)}")

print("\n── Batch × class breakdown ──────────────────────────────────")
batch_class = (
    sample_meta.groupby(["batch", "class"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=list(CLASS_LABEL.keys()), fill_value=0)
)
display(batch_class)

print("\n── Run order range per class ────────────────────────────────")
for cls, grp in sample_meta.groupby("class"):
    print(f"  {cls:8s}  runs {int(grp['order'].min()):>3d} – {int(grp['order'].max()):>3d}")

── Samples per class ────────────────────────────────────────
  Dunn      Control (healthy)               n = 27
  French    Disease (lung cancer)           n = 26
  LMU       Benign (pulmonary disease)      n = 26
  QC        Pooled QC                       n = 22
  B         Blank                           n = 11
  dQC       Diluted QC                      n = 10
  SS        System suitability              n = 2

  Total: 124

── Batch × class breakdown ──────────────────────────────────


class,French,LMU,Dunn,QC,dQC,B,SS
batch,,,,,,,
1,26,26,27,11,5,6,1
2,0,0,0,11,5,5,1



── Run order range per class ────────────────────────────────
  B         runs   1 – 126
  Dunn      runs  15 –  99
  French    runs  16 –  98
  LMU       runs  17 –  96
  QC        runs  12 – 123
  SS        runs   4 – 107
  dQC       runs   7 – 112


## 5 · Missing Value Overview

In LC-MS data, a zero entry means the feature signal was below the instrument detection limit for that sample — it is treated as **missing**, not as a true zero abundance. High missingness can indicate noisy features that will be removed during QC filtering (Phase 2).

Here we report overall and per-class missingness to establish the baseline before any filtering.

In [7]:
is_missing = data_matrix == 0

total_values = data_matrix.size
n_missing    = int(is_missing.values.sum())
pct_missing  = 100 * n_missing / total_values

print(f"── Global missingness ───────────────────────────────────────")
print(f"  Total values          : {total_values:,}")
print(f"  Missing (zero) values : {n_missing:,} ({pct_missing:.1f}%)")
print(f"  Non-missing values    : {total_values - n_missing:,} ({100-pct_missing:.1f}%)")

# Per biological class
bio_classes = ["French", "Dunn", "LMU"]
print(f"\n── Missingness by biological class ──────────────────────────")
for cls in bio_classes:
    idx  = sample_meta[sample_meta["class"] == cls].index
    sub  = data_matrix.loc[idx]
    rate = (sub == 0).values.mean()
    print(f"  {cls:8s} ({CLASS_LABEL[cls]:<28s}): {rate*100:.1f}%  ({len(idx)} samples)")

# Per QC type
print(f"\n── Missingness by QC sample type ────────────────────────────")
for cls in ["QC", "dQC", "B", "SS"]:
    idx = sample_meta[sample_meta["class"] == cls].index
    if len(idx) == 0:
        continue
    sub  = data_matrix.loc[idx]
    rate = (sub == 0).values.mean()
    print(f"  {cls:4s}  ({CLASS_LABEL[cls]:<30s}): {rate*100:.1f}%  ({len(idx)} samples)")

# Feature-level missingness distribution
feat_missing = is_missing.mean()
print(f"\n── Feature-level missingness (across all samples) ───────────")
thresholds = [0.3, 0.5, 0.7, 1.0]
prev = 0.0
for thr in thresholds:
    n = int(((feat_missing > prev) & (feat_missing <= thr)).sum())
    print(f"  Missing {int(prev*100):>3d}–{int(thr*100):>3d}% : {n:>4d} features")
    prev = thr

── Global missingness ───────────────────────────────────────
  Total values          : 31,248
  Missing (zero) values : 6,157 (19.7%)
  Non-missing values    : 25,091 (80.3%)

── Missingness by biological class ──────────────────────────
  French   (Disease (lung cancer)       ): 0.2%  (26 samples)
  Dunn     (Control (healthy)           ): 0.6%  (27 samples)
  LMU      (Benign (pulmonary disease)  ): 1.4%  (26 samples)

── Missingness by QC sample type ────────────────────────────
  QC    (Pooled QC                     ): 35.6%  (22 samples)
  dQC   (Diluted QC                    ): 36.0%  (10 samples)
  B     (Blank                         ): 97.4%  (11 samples)
  SS    (System suitability            ): 84.3%  (2 samples)

── Feature-level missingness (across all samples) ───────────
  Missing   0– 30% :  246 features
  Missing  30– 50% :    5 features
  Missing  50– 70% :    1 features
  Missing  70–100% :    0 features


## 6 · Save Consolidated Output

Bundle the four aligned DataFrames into a single pickle file at `files/data/processed/raw_aligned.pkl`. All downstream notebooks load from this file, so the sample/feature order stays consistent throughout the project.

In [8]:
output = {
    "data_matrix":  data_matrix,   # (n_samples, n_features) — raw intensities, zeros = missing
    "feature_meta": feature_meta,  # (n_features, 10) — m/z and RT columns
    "sample_meta":  sample_meta,   # (n_samples, 4)  — class, id, order, batch
    "standards":    standards,     # (4, 3)           — GU4/GU5/GU14/GU15 reference
    "class_label":  CLASS_LABEL,   # dict             — human-readable class names
}

out_path = OUT_DIR / "raw_aligned.pkl"
with open(out_path, "wb") as f:
    pickle.dump(output, f)

print(f"Saved → {out_path.resolve()}")
print()
for key, val in output.items():
    if isinstance(val, pd.DataFrame):
        print(f"  {key:15s} {val.shape}")
    else:
        print(f"  {key:15s} {type(val).__name__}")

Saved → /Users/hannahhajj/Pictures/AI4chem/files/data/processed/raw_aligned.pkl

  data_matrix     (124, 252)
  feature_meta    (252, 10)
  sample_meta     (124, 4)
  standards       (4, 3)
  class_label     dict
